<a href="https://colab.research.google.com/github/raya-lucaria/ia_o26/blob/main/course/9_aprendizaje_maquina/_assets/01_que_es_f.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab" width="300"></a>

# Notebook 1 · ¿Qué es f? El proceso, el piso y la representación

Acompaña la página [¿Qué es f?](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/que-es-f/). Duración: ~25 min.

Puedes recorrerlo solo en casa, de arriba abajo. Ejecuta primero la celda de
arranque. En cada bloque, escribe tu predicción antes de ejecutar «Juega»;
después mueve los controles, mira el resultado y abre «Explicación».

In [ ]:
#@title ▶ Corre esta celda primero
import os, sys, glob, base64, zlib, importlib, urllib.request
URL = os.environ.get("ML_LAB_URL",
      "https://raw.githubusercontent.com/raya-lucaria/ia_o26/main/cou"
      "rse/9_aprendizaje_maquina/_assets/ml_lab.py")
REQUIERE = "1.5"                     # la versión con que se generó este notebook
_RESPALDO = "..."

def _v(texto): return tuple(int(p) for p in texto.split("."))

def _sirve(version):                 # misma X y una Y mayor o igual
    (x, y), (x0, y0) = _v(version), _v(REQUIERE)
    return x == x0 and y >= y0

def _cargar():                       # importa ml_lab DE CERO (sin la copia en memoria)
    sys.modules.pop("ml_lab", None)
    for viejo in glob.glob("__pycache__/ml_lab.*.pyc"):
        os.remove(viejo)             # un .pyc de la copia vieja taparía la nueva
    importlib.invalidate_caches()
    import ml_lab
    return ml_lab if _sirve(ml_lab.VERSION) else None

ml = None
if os.path.exists("ml_lab.py"):      # 1) la copia local, si ya es nueva
    try: ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 2) la del repositorio
    try:
        urllib.request.urlretrieve(URL, "ml_lab.py"); ml = _cargar()
    except Exception: ml = None
if ml is None:                       # 3) el respaldo: sin red o repo aún sin esta versión
    with open("ml_lab.py", "wb") as archivo:
        archivo.write(zlib.decompress(base64.b64decode(_RESPALDO)))
    ml = _cargar()
ml.requiere(REQUIERE)
ml.tema("claro")
from ml_lab import *
print(f"Listo: ml_lab {ml.VERSION}")

En las entregas, $x$ reúne lo que describes y $y$ es el tiempo observado:
$y=f(x)+\varepsilon$, con $f$ como función del proceso y $\varepsilon$ como ruido,
no como cadena vacía.
Para un predictor $f_\beta$, con parámetros $\beta$, el riesgo
$R(\beta)=E[\ell(y,f_\beta(x))]$ promedia la pérdida $\ell$ en la población;
$E$ indica esperanza. Aquí usamos pérdida cuadrática para los tiempos.
[Proceso generador y Riesgo y error de generalización](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/que-es-f/#raya-object-ml-c1-def-proceso-generador) conectan ambas ecuaciones.

**Predice.** Si cambias la semilla, ¿qué se mueve: los puntos, la línea verde ($f$), la recta ajustada?

Elige una respuesta: solo los puntos; los puntos y la recta; o también la línea
verde. Conserva «proceso» en «entregas», «ruido σ_ε» en 1 y «datos n» en 20.
Mueve únicamente «semilla» y compara la nube, ambas líneas y el riesgo impreso.
Después puedes explorar «recíproca», «mezcla de logaritmos» y «seno» en «proceso».

In [ ]:
#@title Juega: el proceso que no vemos
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 0
PROCESOS_B1 = {"entregas": "entregas", "recíproca": "inverso",
               "mezcla de logaritmos": "mezcla_logs", "seno": "seno"}


def bloque_1(proceso, sigma, n, semilla):
    datos = ml.PROCESOS[proceso](n=n, semilla=semilla, ruido=ml.normal(0, sigma),
                                 fracciones=(1, 0, 0))
    recta = ml.ajustar(datos)
    fig, ax = plt.subplots(figsize=(7, 4.2))
    ml.dibujar_proceso(datos, puntos_realidad=300, ax=ax)
    # Sin datos= para no volver a dibujar (y rotular) los puntos de entrenamiento;
    # el eje muestra tiempos observados: es y, no ŷ.
    ventana = ax.get_xlim()
    ml.dibujar_ajuste({"f̂: la recta ajustada": recta}, ventana=ventana, ax=ax)
    ax.set_xlim(ventana)
    ax.set_ylabel("y")
    plt.show()
    b_ent = ml.benchmarks(datos, datos.ent).loc["media", "cuadratica"]
    b_desp = ml.benchmarks(datos, datos.despliegue).loc["media", "cuadratica"]
    print(f"entrenamiento ({n} datos): L_ent = {ml.L(recta, datos.ent):.3f}"
          f"   ·   la media (benchmark): {b_ent:.3f}")
    print(f"despliegue (100 000 obs.): R(β̂) = {ml.riesgo(recta, datos):.3f}"
          f"   ·   la media (benchmark): {b_desp:.3f}")


ml.laboratorio(
    bloque_1,
    proceso=ml.selector("proceso", PROCESOS_B1),
    sigma=ml.deslizador("ruido σ_ε", 0.0, 3.0, 0.25, 1.0),
    n=ml.deslizador("datos n", 10, 500, 10, 20),
    semilla=ml.deslizador("semilla", 0, 20, 1, SEMILLA),
)

**Mira.** En la gráfica, los círculos son las 20 entregas de entrenamiento; los puntos pequeños son 300 observaciones nuevas.
La línea gruesa verde es $f$; la delgada es la recta ajustada $\widehat f$.

Debajo, `L_ent` mide el error en entrenamiento; la media es el punto de comparación (*benchmark*).
La cifra impresa junto a $R(\widehat\beta)$ aproxima ese riesgo con las 100 000 observaciones del despliegue.

<details><summary>Explicación</summary>

Al mover el control, cambia la semilla: la población y $f$ se quedan; cambian los datos, el ajuste y, con él, su riesgo $R(\widehat\beta)$.

La función $R(\beta)=E[\ell(y,f_\beta(x))]$ permanece fija; cambia el ajuste
$\widehat\beta$ donde la evalúas. El gorro identifica lo estimado con los datos:
$\widehat f=f_{\widehat\beta}$. Es la distinción de
[Lo estimado lleva gorro](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/que-es-f/#raya-object-ml-c1-def-estimacion).

Aquí $L$ denota un error promedio, no un lenguaje. Con los controles iniciales,
la recta tiene $L_{\text{ent}}=0.450$ en
entrenamiento y riesgo aproximado 1.025 en el despliegue, frente a 49.073 de la
media en el despliegue. Esa aproximación usa 100 000 observaciones. En los otros
procesos, la recta no alcanza la forma de $f$: hay un problema de aproximación,
además del azar.

</details>

**Predice.** Si duplicas $\sigma_\varepsilon$ de 1 a 2, ¿el piso se duplica o se cuadruplica?

Anota una de las dos opciones. En «ruido σ_ε», pasa de 1 a 2 y deja «datos n»
en 20. Compara el valor impreso de `R_Bayes` y la altura de la línea punteada.
El oráculo es el predictor que conoce $f$; tampoco puede conocer el ruido de
una entrega nueva.

In [ ]:
#@title Juega: el piso que nadie baja
SEMILLA = 0


class Fijo(ml.Modelo):
    """Un predictor que no se ajusta: predice con una función fija de x."""
    tarea = "regresion"
    def __init__(self, g):
        self.g = g
    def predecir(self, df):
        return np.asarray(self.g(df), dtype=float)


def bloque_2(sigma, n):
    datos = ml.PROCESOS["entregas"](n=n, sigma_e=sigma, semilla=SEMILLA,
                                     fracciones=(1, 0, 0))
    ybar = datos.ent["y"].mean()
    modelos = {"oráculo f": Fijo(lambda d: 10 + 4 * d["x"]),
               "recta ajustada": ml.ajustar(datos),
               "media (benchmark)": Fijo(lambda d: np.full(len(d), ybar))}
    R, valor_piso = {k: ml.riesgo(m, datos) for k, m in modelos.items()}, ml.piso(datos)
    ax = ml.dibujar_piso(datos, modelos, ax=plt.subplots(figsize=(7, 3.6))[1])
    # La media es ~50 veces las otras: su barra se recorta para leer el piso.
    tope = max(5, 1.5 * valor_piso, 1.5 * R["oráculo f"], 1.5 * R["recta ajustada"])
    if R["media (benchmark)"] > tope:
        ax.set_ylim(0, tope)
        ax.text(2, 0.97 * tope, f"↑ media: {R['media (benchmark)']:.1f}\n(recortada)",
                ha="center", va="top", fontsize=9, color="white", fontweight="bold")
    ax.legend(loc="upper left", frameon=False)
    plt.show()
    print(f"piso R_Bayes = σ_ε² = {valor_piso:.3f}")
    for nombre, m in modelos.items():
        print(f"{nombre:>18}: L_ent = {ml.L(m, datos.ent):7.3f}"
              f"   R (despliegue) = {R[nombre]:7.3f}")

ml.laboratorio(
    bloque_2,
    sigma=ml.deslizador("ruido σ_ε", 0.0, 3.0, 0.25, 1.0),
    n=ml.deslizador("datos n", 10, 500, 10, 20),
)

**Mira.** Cada barra muestra el riesgo aproximado en el despliegue del oráculo $f$, la recta ajustada o la media; la línea punteada marca el piso.
La barra del oráculo queda cerca de esa línea, un poco arriba o abajo; ninguna queda claramente por debajo.
Los renglones impresos separan $L_{\text{ent}}$, el error en entrenamiento, de $R$, el riesgo aproximado en el despliegue.

<details><summary>Explicación</summary>

Se cuadruplica: el piso pasa de 1 a 4. En este proceso,
$R_{\text{Bayes}}=\sigma_\varepsilon^2$, como explica
[El piso: error de Bayes](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/que-es-f/#raya-object-ml-c1-def-error-bayes).
La varianza del ruido, $\sigma_\varepsilon^2$, mide su dispersión cuadrática
alrededor de su media cero. El control mueve su desviación estándar
$\sigma_\varepsilon$; duplicarla cuadruplica la varianza.

Ni conocer $f$ permite bajar ese piso en la población. Con ruido 1, el oráculo
imprime un riesgo aproximado de 0.991 en el despliegue: las 100 000
observaciones aproximan la esperanza. Su $L_{\text{ent}}=0.489$ en entrenamiento
tampoco contradice el piso poblacional. Si mueves «datos n», ese error puede
quedar arriba o abajo del piso.

</details>

**Predice.** ¿Qué baja más el error en el despliegue: subir el grado a 5 o medir si llovía?

Elige entre subir «grado N», activar «medir la lluvia x₂» o esperar una mejora
parecida. Primero deja el interruptor apagado y lleva «grado N» de 1 a 5;
anota $R$ en el despliegue. Luego vuelve al grado 1, activa el interruptor y
compara $R$ y el piso.

Aquí $x_1$ es la distancia y $x_2$ indica si llovía. $x_i$, con la letra $i$,
es la observación $i$; $x_1, x_2, \dots$, con número, son **variables**
(columnas); $x_{ij}$ es la variable $j$ de la observación $i$.

In [ ]:
#@title Juega: ruido que no es ruido (la lluvia)
SEMILLA = 0
LLUVIA = {False: ml.caso(2, semilla=SEMILLA),
          True: ml.caso(2, omitir=None, semilla=SEMILLA)}


def _phi_lluvia(medir, N):
    if medir:
        return ml.combinar(ml.polinomio(N, "x1"), ml.columnas("x2"))
    return ml.polinomio(N, "x1")


AJUSTES_B3 = {(medir, N): ml.ajustar(LLUVIA[medir], phi=_phi_lluvia(medir, N))
              for medir in (False, True) for N in range(1, 6)}


def bloque_3(medir, N):
    datos, m = LLUVIA[medir], AJUSTES_B3[(medir, N)]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
    ml.dibujar_proceso(datos, color_por="x2" if medir else None, ax=a1)
    ml.dibujar_piso(datos, {f"grado {N}": m}, ax=a2)
    a2.set_ylim(0, 1.45 * max(ml.riesgo(m, datos), ml.piso(datos)))
    a2.legend(loc="upper center", frameon=False)   # arriba de la barra, no encima
    plt.show()
    media = ml.benchmarks(datos, datos.despliegue).loc["media", "cuadratica"]
    print(f"columnas que ve el modelo: {datos.columnas_x}")
    print(f"piso = {ml.piso(datos):.2f}   ·   "
          f"R (despliegue) = {ml.riesgo(m, datos):.2f}   ·   "
          f"la media (benchmark), R = {media:.2f}")


ml.laboratorio(
    bloque_3,
    medir=ml.interruptor("medir la lluvia x₂", False),
    N=ml.deslizador("grado N", 1, 5, 1, 1),
)

**Mira.** A la izquierda, cada punto es una entrega; al medir la lluvia, los colores distinguen las dos franjas de la nube.
A la derecha, la barra es el riesgo aproximado en el despliegue y la línea punteada es el piso.
Compara también las columnas visibles impresas: al incorporar $x_2$, el piso pasa de 10 a 1.

<details><summary>Explicación</summary>

Medir la lluvia baja más el error en este experimento. Sin medirla, el piso es
10 y el riesgo aproximado con grado 1 es 10.89 en el despliegue; medirla baja el
piso a 1.

Subir el grado no baja el piso y aquí empeora el ajuste: sin medir la lluvia, con grado 5 el riesgo sube de 10.89 a 15.73. Una curva más flexible de la distancia no revela si llovió, y con 20 datos persigue el ruido.

Con pérdida cuadrática, la mejor predicción con lo visible es la media
condicional $E[y\mid x_{\text{visibles}}]$: el tiempo promedio entre entregas
con esos valores visibles. No la confundas con $f$ si omitiste la lluvia.
[La lluvia: ruido que no es ruido](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/que-es-f/#raya-object-ml-c1-ejm-lluvia)
explica por qué el piso depende de lo que mides.

</details>

**Predice.** En cartesianas, ¿la logística pasará de 60 % de aciertos? ¿Y en polares?

Anota sí o no para cada representación. Conserva «radio exterior» en 1.6,
«grosor» en 0.4 y «ruido» en 0.15. En «representación», compara «cartesianas
(1, x₁, x₂)» con «polares (1, r, θ)» mirando los aciertos en el despliegue.
Después elige «solo radio (1, r)» y observa la frontera.

Aquí $x_1,x_2$ son coordenadas del punto; $r$ es su radio, no una recompensa,
y $\theta$ es el ángulo, no un parámetro ajustado. La representación $\phi$
construye las variables que recibe el modelo; la matriz $\Phi$ las reúne, con
una fila por observación.

In [ ]:
#@title Juega: laboratorio polar
SEMILLA = 0
REPRESENTACIONES = {"cartesianas (1, x₁, x₂)": None,
                    "polares (1, r, θ)": ml.polares(),
                    "solo radio (1, r)": ml.polares(incluir=("r",))}

def bloque_4(radio_exterior, grosor, ruido, representacion):
    datos = ml.anillos(n=400, radio_exterior=radio_exterior, grosor=grosor,
                       ruido=ruido, semilla=SEMILLA, fracciones=(1, 0, 0))
    phi, desp = REPRESENTACIONES[representacion], datos.despliegue
    with warnings.catch_warnings(record=True) as avisos:
        warnings.simplefilter("always")
        m = ml.ajustar(datos, phi=phi, tarea="clasificacion")
    if separados := any("separan" in str(a.message) for a in avisos):
        print("Los datos se separan: sin penalización no hay coeficientes óptimos"
              " finitos;\ncrecen hasta que el optimizador se detiene.")
    vista = "cartesianas" if phi is None else "polares"
    ax = ml.dibujar_frontera(m, datos.ent, vista=vista)
    if vista == "polares":   # la frontera ideal: el radio donde Pr(y = 1 | x) cruza 1/2
        r = np.hypot(desp["x1"], desp["x2"]).to_numpy()
        lado = desp["prob"].to_numpy()[r.argsort()] >= 0.5
        r0 = np.sort(r)[np.argmax(lado != lado[0])]
        ax.axhline(r0, ls=":", lw=2, color="gray", label=f"ideal: r = {r0:.2f}")
        ax.legend(loc="upper right")
    plt.show()
    print(m.Phi(datos.ent)[:4]) if separados else ml.imprimir_Phi(m, datos.ent, 4)
    moda_ent = ml.benchmarks(datos, datos.ent).loc["moda", "aciertos"]
    moda = ml.benchmarks(datos, desp).loc["moda", "aciertos"]
    oraculo = np.mean((desp["prob"] >= 0.5).astype(int) == desp["y"])
    print(f"aciertos en ent = {ml.evaluar(m, datos.ent, 'aciertos'):.3f}"
          f"   ·   la moda (benchmark) en ent: {moda_ent:.3f}")
    print(f"aciertos en el despliegue = {ml.desempeno(m, datos, 'aciertos'):.3f}"
          f"   ·   la moda: {moda:.3f}   ·   aciertos del oráculo: {oraculo:.3f}")
    print("coeficientes β̂:", m.beta_gorro.round(3).to_dict())

ml.laboratorio(bloque_4,
               radio_exterior=ml.deslizador("radio exterior", 1.3, 2.2, 0.1, 1.6),
               grosor=ml.deslizador("grosor", 0.3, 0.8, 0.05, 0.4),
               ruido=ml.deslizador("ruido", 0.1, 0.4, 0.05, 0.15),
               representacion=ml.selector("representación", list(REPRESENTACIONES)))

**Mira.** Cada punto es una observación coloreada por su clase; la frontera separa las clases que predice la logística.
En polares, el eje horizontal es $\theta$ y el vertical es $r$: un radio fijo se ve horizontal; la frontera ajustada puede inclinarse un poco.

La línea punteada es la frontera ideal, $r=1.31$: el radio donde las dos clases son igual de probables.
Debajo aparecen cuatro filas de $\Phi$, con «1» primero; compara los aciertos en entrenamiento y en el despliegue con la moda y con los aciertos del oráculo.

<details><summary>Explicación</summary>

Con los valores iniciales, cartesianas no pasa de 60 %: logra 48 % de aciertos
en el despliegue, frente al 50 % de la moda. Polares sí: logra 75 %; solo radio
también logra 75 %. Los aciertos del oráculo en el despliegue son 76 %;
el oráculo conoce las probabilidades del proceso y clasifica con ellas.
Con una muestra finita, esa cifra aproxima el techo de Bayes de la población;
no es una cota exacta para otros clasificadores en esa misma muestra.
La moda predice siempre la clase más frecuente del entrenamiento.

Cambió $\phi$, con la misma logística y el mismo optimizador. Compara sus
columnas: $(1,x_1,x_2)$, $(1,r,\theta)$ y $(1,r)$. La columna de unos permite
un término constante, como en
[Modelo lineal: lineal en sus parámetros](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/que-es-f/#raya-object-ml-c1-def-modelo-lineal).
Con solo radio, la frontera es exactamente horizontal:
$r=-\widehat\beta_0/\widehat\beta_1$. En polares, el coeficiente ajustado de
$\theta$ explica la pequeña inclinación.

Al aumentar «ruido», ni el oráculo llega al 100 % de aciertos en el despliegue.
Si una muestra resulta separable, la celda lo avisa: sin penalización, la
logística no tiene coeficientes óptimos finitos. Crecen hasta que el
optimizador se detiene; esa parada no es un óptimo finito.

</details>

**Predice.** ¿Cuál de estas cuatro $\phi$ necesita un algoritmo distinto de mínimos cuadrados?

Elige una, todas o ninguna: $(1,x)$, $(1,x,x^2)$, $(1,1/x)$ y $(1,x)$ con un
indicador de lluvia. Conserva «proceso» en «entregas con lluvia» y recorre las
opciones de «φ». Compara la forma de la curva, las columnas de $\Phi$ y la
línea `m = ml.ajustar(datos, phi=...)` impresa bajo la gráfica.

In [ ]:
#@title Juega: lineal en β
SEMILLA = 0
DATOS_B5 = {
    "entregas con lluvia": (ml.caso(2, omitir=None, semilla=SEMILLA), "x1"),
    "recíproca": (ml.PROCESOS["inverso"](semilla=SEMILLA, fracciones=(1, 0, 0)), "x"),
}
PHIS_B5 = {"(1, x)": lambda v: ml.columnas(v),
           "(1, x, x²)": lambda v: ml.columnas(v, f"{v}**2"),
           "(1, 1/x)": lambda v: ml.columnas(f"1/{v}"),
           "(1, x) + lluvia": lambda v: ml.combinar(ml.columnas(v), ml.dummies("x2"))}


def bloque_5(proceso, phi):
    datos, v = DATOS_B5[proceso]
    if phi.endswith("lluvia") and "x2" not in datos.ent:
        print("La recíproca no trae lluvia: esta φ no aplica. Elige otra.")
        return
    m = ml.ajustar(datos, phi=PHIS_B5[phi](v))
    curvas = {f"φ = {phi}, sin lluvia" if "x2" in datos.ent else f"φ = {phi}":
              lambda x: m.predecir(pd.DataFrame({v: x, "x2": 0}))}
    if phi.endswith("lluvia"):
        curvas["con lluvia"] = lambda x: m.predecir(pd.DataFrame({v: x, "x2": 1}))
    ml.dibujar_ajuste(curvas, datos).set_ylabel("y")   # datos observados y predicciones
    plt.show()
    print("m = ml.ajustar(datos, phi=...)   ← la misma línea con cualquier φ")
    ml.imprimir_Phi(m, datos.ent, filas=4)
    b_ent = ml.benchmarks(datos, datos.ent).loc["media", "cuadratica"]
    b_desp = ml.benchmarks(datos, datos.despliegue).loc["media", "cuadratica"]
    print(f"entrenamiento: L_ent = {ml.L(m, datos.ent):.3f}"
          f"   ·   la media (benchmark): {b_ent:.3f}")
    print(f"despliegue: R = {ml.riesgo(m, datos):.3f}"
          f"   ·   la media (benchmark): {b_desp:.3f}")
    if phi == "(1, 1/x)" and v == "x1":
        print("Ojo: x₁ llega hasta 0 y 1/x₁ explota; como el coeficiente de 1/x₁ "
              "no es 0,\nel R verdadero de este ajuste es infinito. El número de "
              "arriba solo promedia\nesta muestra enorme.")


ml.laboratorio(bloque_5, proceso=ml.selector("proceso", list(DATOS_B5)),
               phi=ml.selector("φ", list(PHIS_B5)))

**Mira.** En la gráfica, las curvas muestran las predicciones; al incluir el indicador aparecen una curva sin lluvia y otra con lluvia.
Debajo, $\Phi$ muestra las variables transformadas: sus columnas cambian con «φ», mientras la llamada a `ml.ajustar` permanece igual.
Los renglones de errores comparan $L_{\text{ent}}$ en entrenamiento y $R$ aproximado en el despliegue con la media en cada conjunto.

<details><summary>Explicación</summary>

Ninguna necesita otro algoritmo: todas son lineales en $\beta$. Por ejemplo,
$f_\beta(x)=\beta_0+\beta_1x+\beta_2x^2$ curva la predicción en $x$, pero
combina las columnas de $\Phi$ linealmente en sus parámetros. Esa es la
distinción de
[Modelo lineal: lineal en sus parámetros](https://raya-lucaria.github.io/ia_o26/aprendizaje-maquina/el-problema/que-es-f/#raya-object-ml-c1-def-modelo-lineal).

El indicador de lluvia agrega una columna con codificación de base mediante
`ml.dummies("x2")`. Si cambias «proceso» a «recíproca», esa opción no aplica:
el proceso no trae lluvia y la celda lo dice.

La elección de columnas también exige mirar el dominio. En «entregas con
lluvia», la distancia $x_1$ puede acercarse a cero: con $(1,1/x_1)$ este ajuste tiene riesgo verdadero infinito, porque su coeficiente de $1/x_1$ no es cero. El promedio finito impreso sobre observaciones del
despliegue no lo convierte en un riesgo finito; atiende el aviso de la celda.

</details>

**Taller.** Ejecuta el ejemplo activo y después copia una alternativa comentada:
recíproca, seno o lluvia con `ml.caso(2)`. Quita el signo de comentario de todas
las líneas de la alternativa elegida y conserva `ml.taller(datos, clase=1)` al
final. La última asignación a `datos` será la que use el taller.

Mira la nube con $f$ y el panel de errores de los polinomios. Identifica las
etiquetas de entrenamiento, riesgo en el despliegue, piso y media como
benchmark; por ahora basta ubicarlas. Si añades `phi=` a la llamada del taller,
también verás su matriz $\Phi$.

In [ ]:
datos = ml.proceso(n=20, x={"x": ml.uniforme(0, 6)},
                   f=lambda d: 10 + 4 * d.x, ruido=ml.normal(0, 1),
                   fracciones=(1, 0, 0))
# datos = ml.proceso(n=20, x={"x": ml.uniforme(0.3, 3)}, f=lambda d: 2 + 5 / d.x,
#                    ruido=ml.normal(0, 0.5), fracciones=(1, 0, 0))
# datos = ml.proceso(n=50, x={"x": ml.uniforme(0, 2)},
#                    f=lambda d: 4 + 2 * np.sin(5 * d.x),
#                    ruido=ml.normal(0, 0.3), fracciones=(1, 0, 0))
# datos = ml.caso(2)   # una variable que existe y no mides: la lluvia
ml.taller(datos, clase=1)

**Galería.** Pulsa «Caso 1», lee la pregunta que aparece y escribe tu respuesta
antes de abrir «¿Qué se ve?». Haz lo mismo con «Caso 2».
En el caso 1 la celda no imprime coeficientes: antes de abrir «¿Qué se ve?», anota si esperas el coeficiente estimado de $x_1$ mayor, menor o igual que 1, y por qué; luego compáralo con la explicación.
En el caso 1, ignora por ahora la curva de capacidad: la retoma el notebook 02.
En el caso 2 la celda no imprime cifras: para leer el piso y el riesgo $R$ de tu
mejor recta, vuelve a las cifras que imprime el bloque 3 con el interruptor
apagado y el grado 1.

In [ ]:
#@title Galería: casos que se rompen a la vista
ml.galeria(clase=1, casos=[1, 2])

**Ejercicio.** Con $\sigma_\varepsilon=0.5$, ¿cuánto vale el piso? Compruébalo en
el bloque 2. Anota tu cálculo y el valor de `R_Bayes` que imprime la celda.

**Pista:** ¿el deslizador mueve la desviación estándar o su cuadrado?

<details><summary>Explicación</summary>

El piso es $R_{\text{Bayes}}=0.5^2=0.25$. Al poner «ruido σ_ε» en 0.5,
`R_Bayes` imprime 0.250. La barra del oráculo aproxima ese piso con las
observaciones del despliegue; no tiene que coincidir exactamente con la línea.

</details>

**Ejercicio.** Elige la $\phi$ del bloque 5 que da menor $R$ para la recíproca
y explica por qué su $\Phi$ tiene dos columnas. Cambia «proceso» a «recíproca»
y compara el riesgo aproximado en el despliegue de las opciones que aplican.

**Pista:** ¿qué transformación reproduce la forma recíproca y qué columna
permite sumar una constante?

<details><summary>Explicación</summary>

Gana $(1,1/x)$ en esta comparación: permite representar la forma recíproca
del proceso con una constante y un coeficiente que multiplica $1/x$.
$\Phi$ tiene dos columnas, la de unos y la de $1/x$; cada fila corresponde a
una observación. La opción con lluvia no compite porque este proceso no trae
esa variable.

</details>

Te llevas: $f$ no se ve; el piso depende de lo que mides; la representación decide.
La pregunta: ¿por qué no subir el grado hasta que el error sea cero? (notebook 02).